In [1]:
from google.colab import drive


drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path

ZIP_PATH = Path("/content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/food41_yolo.zip")

print("ZIP exists:", ZIP_PATH.exists())

ZIP exists: True


In [3]:
!rm -rf /content/food41_yolo_clean
!unzip -q "/content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/food41_yolo_clean.zip" -d /content/

print("Extracted.")

Extracted.


In [4]:
from pathlib import Path

DATASET = Path("/content/food41_yolo_clean")

print("Dataset:", DATASET.exists())
print("YAML:", (DATASET / "dataset.yaml").exists())

for split in ["train", "val", "test"]:
    images = list((DATASET / "images" / split).glob("*"))
    labels = list((DATASET / "labels" / split).glob("*.txt"))

    print(
        split,
        "images =", len(images),
        "labels =", len(labels)
    )

Dataset: True
YAML: True
train images = 4278 labels = 4278
val images = 1068 labels = 1068
test images = 1362 labels = 1362


In [6]:
!pip install -q -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 7.6 MB/s eta 0:00:00


In [7]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

results = model.train(
    data="/content/food41_yolo_clean/dataset.yaml",

    epochs=80,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,

    patience=15,

    hsv_h=0.015,
    hsv_s=0.60,
    hsv_v=0.40,

    degrees=8.0,
    translate=0.10,
    scale=0.35,
    shear=2.0,
    perspective=0.0005,

    fliplr=0.5,
    flipud=0.0,

    mosaic=0.7,
    mixup=0.05,

    project="/content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/FoodDetectionRuns",
    name="yolov8s_food41_clean_v1",

    save=True,
    plots=True
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics 8.4.142 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/food41_yolo_clean/dataset.yaml, degrees=8.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format

In [8]:

BEST_MODEL = (
    "/content/drive/MyDrive/Colab Notebooks/Deep Learning/"
    "Food Detection/FoodDetectionRuns/"
    "yolov8s_food41_clean_v1/weights/best.pt"
)

model = YOLO(BEST_MODEL)

test_results = model.val(
    data="/content/food41_yolo_clean/dataset.yaml",
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    plots=True
)

Ultralytics 8.4.142 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 11,141,451 parameters, 0 gradients, 28.5 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 10.9±2.8 MB/s, size: 45.5 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/food41_yolo_clean/labels/test... 1362 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1362/1362 470.3it/s 2.9s
val: /content/food41_yolo_clean/images/test/uec_96d062c6ca34e4ecf7f1.jpg: corrupt JPEG restored and saved
val: New cache created: /content/food41_yolo_clean/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 86/86 4.8it/s 18.1s
                   all       1362       1544      0.881      0.841       0.88      0.674
                  rice        118        124      0.938      0.863   

In [9]:
from google.colab import files

uploaded = files.upload()

Saving Screenshot 2026-09-06 at 7.10.59 PM.png to Screenshot 2026-09-06 at 7.10.59 PM.png


In [10]:
from ultralytics import YOLO

model = YOLO(BEST_MODEL)

results = model.predict(
    source="Screenshot 2026-09-06 at 7.10.59 PM.png",
    imgsz=640,
    conf=0.25,
    device=0,
    save=True
)


image 1/1 /content/Screenshot 2026-09-06 at 7.10.59 PM.png: 640x352 (no detections), 28.3ms
Speed: 8.1ms preprocess, 28.3ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 352)
Results saved to /content/runs/detect/predict


In [13]:
results = model.predict(
    source="/content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/Screenshot 2026-09-06 at 7.10.59 PM.png",
    imgsz=640,
    conf=0.01,
    device=0,
    save=True
)


image 1/1 /content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/Screenshot 2026-09-06 at 7.10.59 PM.png: 640x352 2 rices, 1 miso soup, 1 sausage, 2 grilled salmons, 1 green salad, 10.6ms
Speed: 1.8ms preprocess, 10.6ms inference, 1.7ms postprocess per image at shape (1, 3, 640, 352)
Results saved to /content/runs/detect/predict


In [14]:
r = results[0]

if len(r.boxes) == 0:
    print("No detections even at conf=0.01")
else:
    detections = []

    for box in r.boxes:
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])

        detections.append(
            (
                r.names[class_id],
                confidence
            )
        )

    detections.sort(
        key=lambda x: x[1],
        reverse=True
    )

    for name, confidence in detections:
        print(
            f"{name:<20} {confidence:.4f}"
        )

rice                 0.0723
rice                 0.0615
grilled salmon       0.0389
green salad          0.0363
grilled salmon       0.0215
miso soup            0.0126
sausage              0.0115


In [16]:
results = model.predict(
    source="/content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/Screenshot 2026-09-06 at 7.10.59 PM.png",
    imgsz=1280,
    conf=0.01,
    device=0,
    save=True
)


image 1/1 /content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/Screenshot 2026-09-06 at 7.10.59 PM.png: 1280x704 1 rice, 1 toast, 1 miso soup, 2 sausages, 1 grilled salmon, 1 rice ball, 1 banana, 40.6ms
Speed: 3.7ms preprocess, 40.6ms inference, 1.6ms postprocess per image at shape (1, 3, 1280, 704)
Results saved to /content/runs/detect/predict


In [17]:
r = results[0]

detections = []

for box in r.boxes:
    class_id = int(box.cls[0])
    confidence = float(box.conf[0])

    detections.append(
        (r.names[class_id], confidence)
    )

detections.sort(
    key=lambda x: x[1],
    reverse=True
)

for name, confidence in detections:
    print(f"{name:<20} {confidence:.4f}")

sausage              0.6344
rice                 0.1849
grilled salmon       0.0614
sausage              0.0487
toast                0.0233
miso soup            0.0218
banana               0.0136
rice ball            0.0123


In [20]:
results = model.predict(
    source="/content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/‌banana crop.png",
    imgsz=640,
    conf=0.01,
    device=0,
    save=True
)


image 1/1 /content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/‌banana crop.png: 640x480 1 hamburger, 3 hot dogs, 46.8ms
Speed: 1.8ms preprocess, 46.8ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 480)
Results saved to /content/runs/detect/predict


In [21]:
test_images = [
    "/content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/images.jpeg",
    "/content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/images-2.jpeg",
    "/content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/images-3.jpeg"
]

for image_path in test_images:

    print("\n" + "=" * 60)
    print(image_path)
    print("=" * 60)

    results = model.predict(
        source=image_path,
        imgsz=640,
        conf=0.01,
        device=0,
        save=True
    )

    r = results[0]

    detections = []

    for box in r.boxes:
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])

        detections.append(
            (r.names[class_id], confidence)
        )

    detections.sort(
        key=lambda x: x[1],
        reverse=True
    )

    if not detections:
        print("NO DETECTIONS")
    else:
        for name, confidence in detections:
            print(
                f"{name:<20} {confidence:.4f}"
            )


/content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/images.jpeg

image 1/1 /content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/images.jpeg: 448x640 1 french fries, 1 banana, 46.9ms
Speed: 1.9ms preprocess, 46.9ms inference, 1.5ms postprocess per image at shape (1, 3, 448, 640)
Results saved to /content/runs/detect/predict
banana               0.2678
french fries         0.1978

/content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/images-2.jpeg

image 1/1 /content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/images-2.jpeg: 448x640 1 pizza, 1 lasagna, 12.6ms
Speed: 1.6ms preprocess, 12.6ms inference, 1.4ms postprocess per image at shape (1, 3, 448, 640)
Results saved to /content/runs/detect/predict
lasagna              0.6860
pizza                0.1710

/content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/images-3.jpeg

image 1/1 /content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/images-3.jpeg:

In [22]:
base_model = YOLO("yolov8s.pt")

In [23]:
test_images = [
    "/content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/images.jpeg",
    "/content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/images-2.jpeg",
    "/content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/images-3.jpeg"
]

for image_path in test_images:

    print("\n" + "=" * 60)
    print(image_path)
    print("=" * 60)

    results = base_model.predict(
        source=image_path,
        imgsz=640,
        conf=0.01,
        device=0,
        save=False,
        verbose=False
    )

    r = results[0]

    detections = []

    for box in r.boxes:
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])

        detections.append(
            (r.names[class_id], confidence)
        )

    detections.sort(
        key=lambda x: x[1],
        reverse=True
    )

    if not detections:
        print("NO DETECTIONS")
    else:
        for name, confidence in detections:
            print(
                f"{name:<20} {confidence:.4f}"
            )


/content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/images.jpeg
banana               0.9433
person               0.5180
person               0.2641
banana               0.0558
banana               0.0280
tennis racket        0.0228
clock                0.0184
banana               0.0104

/content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/images-2.jpeg
pizza                0.9328
knife                0.0779
knife                0.0705
dining table         0.0688
oven                 0.0564
fork                 0.0404
knife                0.0191
knife                0.0127
knife                0.0117
knife                0.0104
knife                0.0102

/content/drive/MyDrive/Colab Notebooks/Deep Learning/Food Detection/images-3.jpeg
pizza                0.9558
person               0.8960
person               0.2964
person               0.1920
person               0.0596
suitcase             0.0417
person               0.0193
suitcase             0.01